In [0]:
from pyspark.sql.types import StructField, StructType, StringType, TimestampType, IntegerType
from datetime import datetime
import uuid
import time

CONFIG TABLE

In [0]:
api_config_schema = StructType([
    StructField("pipeline_id", StringType(), True),
    StructField("endpoint", StringType(), True),
    StructField("api_key_scope", StringType(), True),
    StructField("search_keyword", StringType(), True),
    StructField("target_table", StringType(), True),
    StructField("landing_path",StringType(), True),
    StructField("is_active", StringType(), True),
    StructField("created_date", TimestampType(), True),

])

spark.createDataFrame([], api_config_schema) \
    .write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("cricket_lakehouse.config.api_config")
print("Table created at: api_config")

Schema for pipeline

In [0]:
pipeline_log_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("pipeline_id", StringType(), True),
    StructField("status", StringType(), True),
    StructField("records_fetched", IntegerType(), True),
    StructField("records_saved", IntegerType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("error_message", StringType(), True),
    StructField("page_size", IntegerType(), True),
    StructField("load_type", StringType(), True)
])

spark.createDataFrame([], schema=pipeline_log_schema) \
    .write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("cricket_lakehouse.config.pipeline_logs")
print("Table created at: cricket_lakehouse.config.pipeline_logs")

In [0]:
%sql
ALTER TABLE cricket_lakehouse.config.pipeline_logs ADD COLUMN Layer string;

In [0]:
from datetime import datetime
import uuid
import time

def run_pipeline_from_config(row):

    run_id = str(uuid.uuid4())
    start_time = datetime.now()

    status = "SUCCESS"
    error_message = None
    records_fetched = 0
    records_saved = 0
    response_code = None
    execution_time_ms = None

    try:
        t0 = time.time()

        # 🔥 EXAMPLE: API CALL (replace with your logic)
        endpoint = row.endpoint
        keyword = row.search_keyword

        # simulate API result
        df = spark.read.json(endpoint)

        records_fetched = df.count()

        # save data
        df.write.mode("append").saveAsTable(row.target_table)

        records_saved = records_fetched

        response_code = 200

        execution_time_ms = int((time.time() - t0) * 1000)

    except Exception as e:
        status = "FAILED"
        error_message = str(e)
        response_code = 500
        execution_time_ms = None

    end_time = datetime.now()

    # 📊 LOG ENTRY
    log_df = spark.createDataFrame([(
        run_id,
        row.pipeline_id,
        status,
        records_fetched,
        records_saved,
        start_time,
        error_message,
        100,
        "full_load",
        end_time,
        response_code,
        execution_time_ms
    )], schema=spark.table("pipeline_logs").schema)

    log_df.write.mode("append").saveAsTable("cricket_lakehouse.config.pipeline_logs")

    return status

In [0]:
def fetch_data():
    return spark.read.json("/path/to/data")

In [0]:
# ═══════════════════════════════════════════════════
# 03_raw_pipeline.py
# PURPOSE  : Generic Metadata Driven Pipeline
# WHO RUNS : Automated / Data Engineer
# WHEN     : Daily
# ═══════════════════════════════════════════════════

import requests
import uuid
from datetime import datetime
from pyspark.sql import functions as F

# ════════════════════════════════════════════════════
# EDIT ONLY THIS SECTION
# ════════════════════════════════════════════════════

CONFIG_TABLE = "cricket_lakehouse.config.api_config"
LOG_TABLE    = "cricket_lakehouse.config.pipeline_logs"

# ════════════════════════════════════════════════════
# DO NOT EDIT BELOW THIS LINE
# Generic code - works for ANY pipeline
# ════════════════════════════════════════════════════

def get_api_key(api_key_scope):
    """
    Generic function
    Gets API key from Databricks Secrets
    Scope name comes from config table
    Works for ANY API!
    """
    return dbutils.secrets.get(
        scope = api_key_scope,
        key   = "api_key"
    )


def call_api(endpoint, api_key,
             search_keyword, http_method):
    """
    Generic function
    Calls ANY API
    Handles pagination
    All params from config table
    """
    all_records = []
    offset      = 0
    page_size   = 10

    while True:

        # Build params
        params = {
            "apikey": api_key,
            "offset": offset,
            "search": search_keyword
        }

        # Call based on http_method
        if http_method == "GET":
            response = requests.get(
                url     = endpoint,
                params  = params,
                timeout = 30
            )
        elif http_method == "POST":
            response = requests.post(
                url     = endpoint,
                json    = params,
                timeout = 30
            )

        response.raise_for_status()
        records = response.json().get("data", [])

        if not records:
            break

        all_records.extend(records)
        print(f"    Offset {offset}: {len(records)} records")

        if len(records) < page_size:
            break

        offset += page_size

    return all_records, response.status_code


def save_to_raw(records, target_table,
                pipeline_id, run_id):
    """
    Generic function
    Saves ANY data to ANY table
    Target table from config table
    """
    df = spark.createDataFrame(records)

    # Add metadata columns
    df = df \
        .withColumn("_pipeline_id", F.lit(pipeline_id)) \
        .withColumn("_run_id",      F.lit(run_id)) \
        .withColumn("_load_time",   F.current_timestamp())

    df.write \
      .format("delta") \
      .mode("overwrite") \
      .option("mergeSchema", "true") \
      .saveAsTable(target_table)

    return df.count()


def write_log(log_table, run_id, pipeline_id,
              status, records_fetched=0,
              records_saved=0, start_time=None,
              end_time=None, response_code=None,
              error_message=None):
    """
    Generic logger
    Logs ANY pipeline run
    """
    log_data = [{
        "run_id":                run_id,
        "pipeline_id":           pipeline_id,
        "status":                status,
        "records_fetched":       records_fetched,
        "records_saved":         records_saved,
        "start_time":            start_time,
        "end_time":              end_time,
        "error_message":         error_message,
        "page_size":             10,
        "load_type":             "overwrite",
        "response_code":         response_code,
        "api_execution_time_ms": None
    }]

    spark.createDataFrame(log_data) \
         .write \
         .format("delta") \
         .mode("append") \
         .saveAsTable(log_table)


def run_pipeline(config_table, log_table):
    """
    Main generic pipeline
    Reads EVERYTHING from config table
    Works for ANY API
    ZERO hardcoding!
    """

    print("═" * 50)
    print("CRICKET LAKEHOUSE - RAW PIPELINE")
    print("═" * 50)

    # ── Read metadata table ──────────────────────────
    config_list = spark.table(config_table) \
                       .filter(F.col("is_active") == "Y") \
                       .collect()

    print(f"\nActive pipelines : {len(config_list)}")
    print("─" * 50)

    # ── Process each pipeline ────────────────────────
    for config in config_list:

        # Read ALL from config table
        # NOTHING hardcoded!
        pipeline_id    = config["pipeline_id"]
        endpoint       = config["endpoint"]
        api_key_scope  = config["api_key_scope"]
        search_keyword = config["search_keyword"]
        target_table   = config["target_table"]
        http_method    = config["http_method"]

        run_id     = str(uuid.uuid4())
        start_time = datetime.now()

        print(f"\n▶ Pipeline  : {pipeline_id}")
        print(f"  Endpoint  : {endpoint}")
        print(f"  Search    : {search_keyword}")
        print(f"  Target    : {target_table}")
        print(f"  Method    : {http_method}")

        try:
            # Step 1: Get API key from secrets
            api_key = get_api_key(api_key_scope)
            print(f"  API Key   : ✅ from secrets")

            # Step 2: Call API
            records, response_code = call_api(
                endpoint       = endpoint,
                api_key        = api_key,
                search_keyword = search_keyword,
                http_method    = http_method
            )
            print(f"  Fetched   : {len(records)} records")

            if not records:
                print(f"  ⚠️ No data returned")
                continue

            # Step 3: Save to raw table
            records_saved = save_to_raw(
                records      = records,
                target_table = target_table,
                pipeline_id  = pipeline_id,
                run_id       = run_id
            )

            end_time = datetime.now()
            duration = (end_time - start_time).seconds

            print(f"  Saved     : {records_saved} records")
            print(f"  Duration  : {duration} seconds")
            print(f"  ✅ SUCCESS")

            # Step 4: Log success
            write_log(
                log_table       = log_table,
                run_id          = run_id,
                pipeline_id     = pipeline_id,
                status          = "success",
                records_fetched = len(records),
                records_saved   = records_saved,
                start_time      = start_time,
                end_time        = end_time,
                response_code   = response_code
            )

        except Exception as e:

            end_time = datetime.now()
            print(f"  ❌ FAILED : {str(e)}")

            # Log failure
            write_log(
                log_table     = log_table,
                run_id        = run_id,
                pipeline_id   = pipeline_id,
                status        = "failed",
                start_time    = start_time,
                end_time      = end_time,
                error_message = str(e)
            )

    print("\n" + "═" * 50)
    print("✅ PIPELINE COMPLETE!")
    print("═" * 50)

    # Show results
    print("\nPipeline Log:")
    spark.table(log_table).show(truncate=False)

    print("\nRaw Tables Created:")
    spark.sql(
        "SHOW TABLES IN cricket_lakehouse.raw_layer"
    ).show()


# ── Run ──────────────────────────────────────────────
run_pipeline(
    config_table = CONFIG_TABLE,
    log_table    = LOG_TABLE
)